# Оптимизация сети Джексона с ограничением бюджета  
**Студент:** *Перминов Н.А.*
В работе рассматривается открытая сеть серверов (Jackson‑сеть) со следующими характеристиками:  
- **Модели очередей** – FIFO или Processor Sharing (PS).  
- **Скорость обслуживания** `µ` и внешняя интенсивность поступления `λ` известны для узла `chain1`.  
- Межузловые переходы заданы вероятностями.  
Наша цель – *оптимизировать взвешенный параметр*  
$$
\Phi = \frac{\text{пропускная способность (throughput)}}{\text{средняя задержка}}
$$
путём добавления 1–2 новых серверов, при этом суммарные затраты на новые узлы не должны превышать  
10% от исходной стоимости сети.  
Для анализа используется **теорема Джексона**: для открытой сети с независимыми M/M/1‑очередями
статистическое распределение в состоянии системы имеет произведённую форму, а пропускная способность
не зависит от дисциплины обслуживания (FIFO↔PS).  
Это позволяет оценивать влияние изменения числа серверов без детального моделирования.
Ниже приведён Jupyter‑Notebook, реализующий все расчёты и выводы.  

In [1]:
import yaml
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

## Загрузка конфигурации сети из YAML

In [2]:
yaml_str = """
servers:
  - id: chain1
    mu: 5.27253214380333
    queue_type: FIFO
    lambda: 2.636266071901665
  - id: chain2
    mu: 5.83754313849087
    queue_type: PS
  - id: chain3
    mu: 4.549136880106992
    queue_type: FIFO
  - id: chain4
    mu: 8.123987347260954
    queue_type: PS
transitions:
  - from: chain1
    to: exit
    probability: 0.5131491852873884
  - from: chain1
    to: chain2
    probability: 0.1973396059215003
  - from: chain1
    to: chain4
    probability: 0.2895112087911113
  - from: chain2
    to: exit
    probability: 0.6076484690150405
  - from: chain2
    to: chain3
    probability: 0.2976274347518623
  - from: chain2
    to: chain1
    probability: 0.09472409623309719
  - from: chain3
    to: exit
    probability: 0.6284346346305989
  - from: chain3
    to: chain2
    probability: 0.1016264474817383
  - from: chain3
    to: chain1
    probability: 0.190476652223234
  - from: chain3
    to: chain4
    probability: 0.07946226566442882
  - from: chain4
    to: exit
    probability: 0.8016734669231556
  - from: chain4
    to: chain1
    probability: 0.1517788797562842
  - from: chain4
    to: chain2
    probability: 0.04654765332056024
"""
config = yaml.safe_load(yaml_str)

In [3]:
# config

## Парсинг результатов симуляции

In [4]:
sim_text = """
Server 'chain1'
  Type: FIFO
  Cost: 14899.8
  Jobs: 2818
  Avg queue time: 0.205557
  Avg service time: 0.187578
  Utilization: 0.528594
Server 'chain2'
  Type: PS
  Cost: 18038.5
  Jobs: 613
  Avg queue time: 0.0109543
  Avg service time: 0.162924
  Utilization: 0.0998724
Server 'chain3'
  Type: FIFO
  Cost: 11347.3
  Jobs: 186
  Avg queue time: 0.00526057
  Avg service time: 0.223484
  Utilization: 0.041568
Server 'chain4'
  Type: PS
  Cost: 33999.6
  Jobs: 853
  Avg queue time: 0.0118444
  Avg service time: 0.118478
  Utilization: 0.101062
Total system cost: 78285.2
Total tasks: 2520
Mean time: 0.51617
"""
def parse_simulation(text):
    server_blocks = text.strip().split("Server")
    data = {}
    for block in server_blocks[1:]:
        # id
        m_id = re.search(r"'([^']+)'", block)
        if not m_id:
            continue
        srv_id = m_id.group(1)
        # key-value pairs
        kvs = {}
        for line in block.splitlines():
            m = re.match(r"\s*(.+?):\s*(.*)", line)
            if m:
                k, v = m.group(1).strip(), m.group(2)
                kvs[k] = v
        # convert to proper types
        parsed = {}
        for k, v in kvs.items():
            if k.strip() == 'Type':
                parsed['Type'] = v.strip()
            elif k.strip() == 'Cost':
                parsed['Cost'] = float(v)
            elif k.strip() == 'Jobs':
                parsed['Jobs'] = int(v)
            elif k.strip() == 'Avg queue time':
                parsed['Avg_queue_time'] = float(v)
            elif k.strip() == 'Avg service time':
                parsed['Avg_service_time'] = float(v)
            elif k.strip() == 'Utilization':
                parsed['Utilization'] = float(v)
        data[srv_id] = parsed
    # parse global metrics
    total_cost = re.search(r"Total system cost:\s*([-\d\.]+)", text).group(1)
    total_tasks = re.search(r"Total tasks:\s*(\d+)", text).group(1)
    mean_time = re.search(r"Mean time:\s*([-\d\.]+)", text).group(1)
    return data, float(total_cost), int(total_tasks), float(mean_time)
sim_data, total_cost, total_tasks, mean_sojourn = parse_simulation(sim_text)
print(f"Total system cost:  {total_cost:.3f}")
print(f"Total tasks:        {total_tasks:.3f}")
print(f"Mean time:          {mean_sojourn:.3f}")

Total system cost:  78285.200
Total tasks:        2520.000
Mean time:          0.516


In [5]:
pd.DataFrame.from_dict(sim_data, orient='index')

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization
chain1,FIFO,14899.8,2818,0.205557,0.187578,0.528594
chain2,PS,18038.5,613,0.010954,0.162924,0.099872
chain3,FIFO,11347.3,186,0.005261,0.223484,0.041568
chain4,PS,33999.6,853,0.011844,0.118478,0.101062


In [6]:
for srv in config['servers']:
    sid = srv['id']
    data = sim_data.get(sid)
    if data:
        srv['Cost'] = data['Cost']
        srv['Jobs'] = data['Jobs']
        srv['Avg_queue_time'] = data['Avg_queue_time']
        srv['Avg_service_time'] = data['Avg_service_time']
        srv['Utilization'] = data['Utilization']
# config

## Основные метрики сети
### Пропускная способность и средняя задержка
*Пропускная способность* – это количество задач, обработанных за единицу времени  
(средний throughput). Поскольку симуляция проводилась в течение `max time = 1000` секунд,  
$$
\lambda_{\text{net}}=\frac{\text{Total tasks}}{\text{Sim. time}}
$$
*Средняя задержка* – это уже заданный показатель mean sojourn time.

In [7]:
sim_time = 1000  # как указано в выводе симуляции
throughput_net = total_tasks / sim_time
mean_delay_net = mean_sojourn
print(f"Throughput (tasks/s):  {throughput_net:.3f}")
print(f"Mean delay (s):        {mean_delay_net:.5f}")
# Взвешенный параметр Φ
phi_net = throughput_net / mean_delay_net
print(f"Φ (throughput/delay):  {phi_net:.3f}")

Throughput (tasks/s):  2.520
Mean delay (s):        0.51617
Φ (throughput/delay):  4.882


### Таблица по серверам

In [8]:
df_srv = pd.DataFrame.from_dict(sim_data, orient='index')
df_srv['mu'] = df_srv.index.map(lambda x: next((s['mu'] for s in config['servers'] if s['id'] == x), np.nan))
df_srv['lambda_ext'] = None
# только chain1 имеет внешнюю интенсивность λ в конфиге
for srv in config['servers']:
    if 'lambda' in srv:
        df_srv.at[srv['id'], 'lambda_ext'] = srv['lambda']
df_srv[['Type', 'Cost', 'Jobs', 'Avg_queue_time', 'Avg_service_time', 'Utilization', 'mu', 'lambda_ext']]

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization,mu,lambda_ext
chain1,FIFO,14899.8,2818,0.205557,0.187578,0.528594,5.272532,2.636266
chain2,PS,18038.5,613,0.010954,0.162924,0.099872,5.837543,None
chain3,FIFO,11347.3,186,0.005261,0.223484,0.041568,4.549137,None
chain4,PS,33999.6,853,0.011844,0.118478,0.101062,8.123987,None


## Оценка стоимости добавления новых серверов
Для каждого типа сервера вычислим **затраты на единицу скорости обслуживания**:
$$
c_{\mu} = \frac{\text{Cost}}{\mu}
$$
Это позволяет понять, какой тип узла наиболее экономичен.

In [9]:
cost_mu = {}
for srv in config['servers']:
    cost_mu[srv['id']] = srv['Cost'] / srv['mu'] 
cost_mu

{'chain1': 2825.928717667724,
 'chain2': 3090.0842309943664,
 'chain3': 2494.3852645148636,
 'chain4': 4185.087758840878}

### Бюджет

In [10]:
budget = 0.10 * total_cost   # 10 % от исходной стоимости
print(f"Бюджет на новые серверы: {budget:.2f}")


Бюджет на новые серверы: 7828.52


## Анализ текущей сети: уравнения трафика и узкие места

In [11]:
import numpy as np
from copy import deepcopy

def solve_traffic(servers, transitions):
    """
    Решает уравнения трафика для открытой сети Джексона.
    Возвращает словарь lambda_eff (интенсивности), матрицу переходов P (только между серверами),
    внешние интенсивности lambda_ext, списки ids и словарь idx.
    """
    n = len(servers)
    ids = [s['id'] for s in servers]
    idx = {id_: i for i, id_ in enumerate(ids)}

    # Матрица переходов P (n x n) между серверами
    P = np.zeros((n, n))
    # Внешний поток
    lambda_ext = np.zeros(n)
    for s in servers:
        if 'lambda' in s:
            lambda_ext[idx[s['id']]] = s['lambda']

    for t in transitions:
        from_id = t['from']
        to_id = t['to']
        if to_id == 'exit':
            continue  # выход из сети, не добавляем в матрицу переходов
        i = idx[from_id]
        j = idx[to_id]
        P[i, j] = t['probability']

    # Решаем (I - P^T) * lambda = lambda_ext
    A = np.eye(n) - P.T
    lambda_eff = np.linalg.solve(A, lambda_ext)

    # Словарь для удобства
    lambda_dict = {ids[i]: lambda_eff[i] for i in range(n)}
    return lambda_dict, P, lambda_ext, ids, idx

def compute_metrics(servers, lambda_eff):
    """
    Вычисляет утилизацию ρ, среднее время пребывания E[T],
    коэффициент посещения V (относительно внешнего потока),
    вклад в общее время и общее E[T_sys].
    """
    metrics = {}
    total_lambda_ext = sum(s.get('lambda', 0) for s in servers)
    for s in servers:
        sid = s['id']
        mu = s['mu']
        lam = lambda_eff[sid]
        rho = lam / mu
        E_T = 1.0 / (mu - lam) if mu > lam else np.inf
        # Среднее время обслуживания и очереди
        E_S = 1.0 / mu
        E_W = E_T - E_S
        metrics[sid] = {
            'lambda': lam,
            'mu': mu,
            'rho': rho,
            'E_T': E_T,
            'E_S': E_S,
            'E_W': E_W,
        }
    # Коэффициенты посещения и общее время
    V = {}
    for s in servers:
        sid = s['id']
        V[sid] = metrics[sid]['lambda'] / total_lambda_ext
    E_T_sys = sum(V[sid] * metrics[sid]['E_T'] for sid in metrics)
    for sid in metrics:
        metrics[sid]['V'] = V[sid]
        metrics[sid]['contrib'] = V[sid] * metrics[sid]['E_T']
    return metrics, E_T_sys, total_lambda_ext

# Применяем к исходной конфигурации
lambda_eff, P, lambda_ext, ids, idx = solve_traffic(config['servers'], config['transitions'])
metrics, E_T_sys, total_lambda_ext = compute_metrics(config['servers'], lambda_eff)

print("=== Текущие эффективные интенсивности λ_i ===")
for sid in metrics:
    print(f"{sid}: λ = {metrics[sid]['lambda']:.4f}, ρ = {metrics[sid]['rho']:.4f}, E[T] = {metrics[sid]['E_T']:.4f}, V = {metrics[sid]['V']:.4f}, вклад = {metrics[sid]['contrib']:.4f}")
print(f"\nОбщая интенсивность внешнего потока: {total_lambda_ext:.4f}")
print(f"Теоретическое среднее время пребывания E[T_sys] = {E_T_sys:.4f} (симуляция: {mean_sojourn:.4f})")

=== Текущие эффективные интенсивности λ_i ===
chain1: λ = 2.8583, ρ = 0.5421, E[T] = 0.4142, V = 1.0842, вклад = 0.4491
chain2: λ = 0.6221, ρ = 0.1066, E[T] = 0.1917, V = 0.2360, вклад = 0.0452
chain3: λ = 0.1851, ρ = 0.0407, E[T] = 0.2291, V = 0.0702, вклад = 0.0161
chain4: λ = 0.8422, ρ = 0.1037, E[T] = 0.1373, V = 0.3195, вклад = 0.0439

Общая интенсивность внешнего потока: 2.6363
Теоретическое среднее время пребывания E[T_sys] = 0.5543 (симуляция: 0.5162)


In [12]:
# Определим узел с наибольшим вкладом в задержку
bottleneck = max(metrics.items(), key=lambda x: x[1]['contrib'])
print(f"\nБутылочное горлышко: {bottleneck[0]} (вклад {bottleneck[1]['contrib']:.4f} из {E_T_sys:.4f})")


Бутылочное горлышко: chain1 (вклад 0.4491 из 0.5543)


## Модель стоимости серверов

In [13]:
def cost_model(servers, queue_type):
    """
    По серверам заданного типа оценивает коэффициенты c и α в cost = c * μ^α.
    Возвращает (c, alpha).
    """
    # Собираем пары (mu, cost) для серверов нужного типа
    data = [(s['mu'], s['Cost']) for s in servers if s['queue_type'] == queue_type]
    if len(data) < 2:
        raise ValueError(f"Недостаточно серверов типа {queue_type} для построения модели")
    mu1, cost1 = data[0]
    mu2, cost2 = data[1]
    alpha = np.log(cost2 / cost1) / np.log(mu2 / mu1)
    c = cost1 / (mu1 ** alpha)
    return c, alpha

# Для FIFO и PS
c_fifo, alpha_fifo = cost_model(config['servers'], 'FIFO')
c_ps, alpha_ps = cost_model(config['servers'], 'PS')
print(f"FIFO: cost = {c_fifo:.2f} * μ^{alpha_fifo:.3f}")
print(f"PS:   cost = {c_ps:.2f} * μ^{alpha_ps:.3f}")

# Бюджет уже вычислен
print(f"\nБюджет на новые серверы: {budget:.2f} (10% от {total_cost:.2f})")

FIFO: cost = 692.77 * μ^1.846
PS:   cost = 612.01 * μ^1.918

Бюджет на новые серверы: 7828.52 (10% от 78285.20)


## Подбор вариантов стратегий

In [14]:
from copy import deepcopy
import numpy as np
from scipy.optimize import brentq

def add_parallel_server(config, target, mu_new, alpha):
    """
    Добавляет один PS‑сервер параллельно узлу target.
    """
    new_servers = deepcopy(config['servers'])
    new_transitions = deepcopy(config['transitions'])

    target_server = next(s for s in new_servers if s['id'] == target)
    lambda_ext_orig = target_server.get('lambda', 0.0)

    new_id = f"{target}_p"
    new_server = {
        'id': new_id,
        'mu': mu_new,
        'queue_type': 'PS',
    }
    if lambda_ext_orig > 0:
        new_server['lambda'] = alpha * lambda_ext_orig
        target_server['lambda'] = (1 - alpha) * lambda_ext_orig
    new_servers.append(new_server)

    # Переходы на target
    for trans in new_transitions:
        if trans['to'] == target:
            p_old = trans['probability']
            trans['probability'] = (1 - alpha) * p_old
            new_transitions.append({
                'from': trans['from'],
                'to': new_id,
                'probability': alpha * p_old
            })

    # Исходящие переходы от нового сервера
    outgoing = [t for t in config['transitions'] if t['from'] == target]
    for t in outgoing:
        new_transitions.append({
            'from': new_id,
            'to': t['to'],
            'probability': t['probability']
        })

    return {'servers': new_servers, 'transitions': new_transitions}

def add_two_parallel_same_node(config, target, mu1, mu2, alpha1, alpha2):
    """
    Добавляет два PS‑сервера параллельно одному узлу target.
    alpha1+alpha2 <= 1, доля исходного = 1-alpha1-alpha2.
    """
    new_servers = deepcopy(config['servers'])
    new_transitions = deepcopy(config['transitions'])

    target_server = next(s for s in new_servers if s['id'] == target)
    lambda_ext_orig = target_server.get('lambda', 0.0)

    new_ids = []
    for i, (mu, alpha) in enumerate([(mu1, alpha1), (mu2, alpha2)]):
        new_id = f"{target}_p{i+1}"
        new_server = {
            'id': new_id,
            'mu': mu,
            'queue_type': 'PS',
        }
        if lambda_ext_orig > 0:
            new_server['lambda'] = alpha * lambda_ext_orig
        new_servers.append(new_server)
        new_ids.append((alpha, new_id))

    if lambda_ext_orig > 0:
        target_server['lambda'] = (1 - alpha1 - alpha2) * lambda_ext_orig

    # Переходы на target
    for trans in new_transitions:
        if trans['to'] == target:
            p_old = trans['probability']
            trans['probability'] = (1 - alpha1 - alpha2) * p_old
            for alpha, new_id in new_ids:
                new_transitions.append({
                    'from': trans['from'],
                    'to': new_id,
                    'probability': alpha * p_old
                })

    # Исходящие переходы от новых серверов
    outgoing = [t for t in config['transitions'] if t['from'] == target]
    for alpha, new_id in new_ids:
        for t in outgoing:
            new_transitions.append({
                'from': new_id,
                'to': t['to'],
                'probability': t['probability']
            })

    return {'servers': new_servers, 'transitions': new_transitions}

def evaluate_config(config, original_total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps):
    """Вычисляет E[T_sys] и стоимость добавленных серверов."""
    lambda_eff, _, _, _, _ = solve_traffic(config['servers'], config['transitions'])
    _, E_T_sys, _ = compute_metrics(config['servers'], lambda_eff)

    total_cost_new = 0
    for s in config['servers']:
        if s['queue_type'] == 'FIFO':
            total_cost_new += c_fifo * (s['mu'] ** alpha_fifo)
        else:
            total_cost_new += c_ps * (s['mu'] ** alpha_ps)
    added_cost = total_cost_new - original_total_cost
    return E_T_sys, added_cost

# --- Параметры моделей стоимости ---
c_fifo, alpha_fifo = cost_model(config['servers'], 'FIFO')
c_ps, alpha_ps = cost_model(config['servers'], 'PS')
print(f"Модели стоимости: FIFO: {c_fifo:.2f}*μ^{alpha_fifo:.3f}, PS: {c_ps:.2f}*μ^{alpha_ps:.3f}")

# --- Ранжирование узлов по нагрузке (вклад в E[T_sys]) ---
candidates = [(sid, metrics[sid]['contrib']) for sid in metrics if metrics[sid]['lambda'] > 1e-6]
candidates.sort(key=lambda x: x[1], reverse=True)  # по убыванию вклада
print("Узлы-кандидаты (вклад в общую задержку):")
for sid, contrib in candidates[:3]:
    print(f"  {sid}: contrib={contrib:.4f}, ρ={metrics[sid]['rho']:.4f}, E[T]={metrics[sid]['E_T']:.4f}")

best_E_T = E_T_sys
best_config = None
best_desc = "исходная конфигурация"

# --- Сценарий 1: один новый сервер к лучшему узлу ---
target1 = candidates[0][0]
Lambda1 = metrics[target1]['lambda']
mu_A1 = metrics[target1]['mu']

# Максимально возможная μ за весь бюджет
mu_max1 = (budget / c_ps) ** (1.0 / alpha_ps)
if mu_max1 > 0:
    # Оптимальная доля на новый сервер
    alpha1 = (mu_max1 - mu_A1 + Lambda1) / (2 * Lambda1)
    alpha1 = np.clip(alpha1, 0.0, 1.0)
    # Если alpha1 близка к 0 или 1, корректируем μ, чтобы получить разумное разделение
    if alpha1 < 0.1:
        # Слишком малая доля → выгоднее направить больше трафика на новый сервер, снизив μ
        alpha1 = 0.5  # желаемая доля
        # Из уравнения alpha = (μ - μ_A + Λ)/(2Λ) => μ = μ_A + (2α-1)Λ
        mu_max1 = mu_A1 + (2 * alpha1 - 1) * Lambda1
        if mu_max1 <= 0:
            mu_max1 = 0.1
        # Проверяем стоимость
        cost1 = c_ps * (mu_max1 ** alpha_ps)
        if cost1 > budget:
            mu_max1 = (budget / c_ps) ** (1.0 / alpha_ps)
            alpha1 = (mu_max1 - mu_A1 + Lambda1) / (2 * Lambda1)
            alpha1 = np.clip(alpha1, 0.0, 1.0)
    elif alpha1 > 0.9:
        alpha1 = 0.5
        mu_max1 = mu_A1 + (2 * alpha1 - 1) * Lambda1
        if mu_max1 <= 0:
            mu_max1 = 0.1
        cost1 = c_ps * (mu_max1 ** alpha_ps)
        if cost1 > budget:
            mu_max1 = (budget / c_ps) ** (1.0 / alpha_ps)
            alpha1 = (mu_max1 - mu_A1 + Lambda1) / (2 * Lambda1)
            alpha1 = np.clip(alpha1, 0.0, 1.0)

    config1 = add_parallel_server(config, target1, mu_max1, alpha1)
    E_T1, added1 = evaluate_config(config1, total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps)
    if added1 <= budget + 1e-6 and E_T1 < best_E_T:
        best_E_T = E_T1
        best_config = config1
        best_desc = f"1 сервер к {target1} (μ={mu_max1:.4f}, α={alpha1:.4f})"

# --- Сценарий 2: два сервера к одному узлу (лучшему) ---
if len(candidates) >= 1:
    target2 = candidates[0][0]
    Lambda2 = metrics[target2]['lambda']
    mu_A2 = metrics[target2]['mu']
    # Делим бюджет поровну на два новых сервера (оба с одинаковой μ)
    mu_each = (budget / (2 * c_ps)) ** (1.0 / alpha_ps)
    if mu_each > 0:
        # Ищем симметричное разделение α1=α2=α
        def f_alpha(alpha):
            if alpha <= 0 or alpha >= 0.5:
                return 1e6
            denom_old = mu_A2 - (1 - 2*alpha) * Lambda2
            denom_new = mu_each - alpha * Lambda2
            if denom_old <= 0 or denom_new <= 0:
                return 1e6
            return denom_new - denom_old / np.sqrt(2)
        try:
            alpha_opt = brentq(f_alpha, 0.01, 0.49)
        except:
            alpha_opt = 1.0/3.0
        alpha_opt = np.clip(alpha_opt, 0.0, 0.5)
        # Проверка стоимости
        cost2 = 2 * c_ps * (mu_each ** alpha_ps)
        if cost2 > budget:
            mu_each = (budget / (2 * c_ps)) ** (1.0 / alpha_ps)
        config2 = add_two_parallel_same_node(config, target2, mu_each, mu_each, alpha_opt, alpha_opt)
        E_T2, added2 = evaluate_config(config2, total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps)
        if added2 <= budget + 1e-6 and E_T2 < best_E_T:
            best_E_T = E_T2
            best_config = config2
            best_desc = f"2 сервера к {target2} (μ={mu_each:.4f} каждый, α={alpha_opt:.4f})"

# --- Сценарий 3: два сервера к двум лучшим узлам (по одному) ---
if len(candidates) >= 2:
    targets = [candidates[0][0], candidates[1][0]]
    contribs = [candidates[0][1], candidates[1][1]]
    total_contrib = sum(contribs)
    # Распределяем бюджет пропорционально вкладу в задержку
    budgets = [budget * (c / total_contrib) for c in contribs]

    new_config_temp = config  # начинаем с исходной
    added_total = 0
    ok = True
    for (target, b) in zip(targets, budgets):
        Lambda = metrics[target]['lambda']
        mu_A = metrics[target]['mu']
        mu_max = (b / c_ps) ** (1.0 / alpha_ps)
        if mu_max <= 0:
            ok = False
            break
        alpha = (mu_max - mu_A + Lambda) / (2 * Lambda)
        alpha = np.clip(alpha, 0.0, 1.0)
        # Коррекция при экстремальных alpha
        if alpha < 0.1 or alpha > 0.9:
            alpha = 0.5
            mu_max = mu_A + (2*alpha - 1) * Lambda
            if mu_max <= 0:
                mu_max = 0.1
            cost_tmp = c_ps * (mu_max ** alpha_ps)
            if cost_tmp > b:
                mu_max = (b / c_ps) ** (1.0 / alpha_ps)
                alpha = (mu_max - mu_A + Lambda) / (2 * Lambda)
                alpha = np.clip(alpha, 0.0, 1.0)
        new_config_temp = add_parallel_server(new_config_temp, target, mu_max, alpha)
        _, added = evaluate_config(new_config_temp, total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps)
        added_total = added  # накопительная стоимость
        if added_total > budget + 1e-6:
            ok = False
            break
    if ok:
        E_T3, added3 = evaluate_config(new_config_temp, total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps)
        if added3 <= budget + 1e-6 and E_T3 < best_E_T:
            best_E_T = E_T3
            best_config = new_config_temp
            best_desc = f"2 сервера: к {targets[0]} и к {targets[1]} (бюджет распределён пропорционально вкладу)"

print("\n" + "="*60)
print("ЛУЧШИЙ ВАРИАНТ (интеллектуальный выбор)")
print("="*60)
print(best_desc)
print(f"Теоретическое E[T_sys]: {E_T_sys:.4f} → {best_E_T:.4f} (улучшение {(E_T_sys - best_E_T)/E_T_sys*100:.2f}%)")

# Сохраняем лучшую конфигурацию
new_config = best_config

Модели стоимости: FIFO: 692.77*μ^1.846, PS: 612.01*μ^1.918
Узлы-кандидаты (вклад в общую задержку):
  chain1: contrib=0.4491, ρ=0.5421, E[T]=0.4142
  chain2: contrib=0.0452, ρ=0.1066, E[T]=0.1917
  chain4: contrib=0.0439, ρ=0.1037, E[T]=0.1373

ЛУЧШИЙ ВАРИАНТ (интеллектуальный выбор)
1 сервер к chain1 (μ=3.7774, α=0.2385)
Теоретическое E[T_sys]: 0.5543 → 0.4554 (улучшение 17.84%)


## Теоретический расчёт для выбранной конфигурации

In [15]:
lambda_eff_new, _, _, _, _ = solve_traffic(new_config['servers'], new_config['transitions'])
metrics_new, E_T_sys_new, _ = compute_metrics(new_config['servers'], lambda_eff_new)

print("=== Результаты после оптимизации ===")
for sid in metrics_new:
    m = metrics_new[sid]
    print(f"{sid}: λ={m['lambda']:.4f}, ρ={m['rho']:.4f}, E[T]={m['E_T']:.4f}, вклад={m['contrib']:.4f}")
print(f"\nНовое теоретическое среднее время E[T_sys] = {E_T_sys_new:.4f}")

=== Результаты после оптимизации ===
chain1: λ=2.1767, ρ=0.4128, E[T]=0.3230, вклад=0.2667
chain2: λ=0.6221, ρ=0.1066, E[T]=0.1917, вклад=0.0452
chain3: λ=0.1851, ρ=0.0407, E[T]=0.2291, вклад=0.0161
chain4: λ=0.8422, ρ=0.1037, E[T]=0.1373, вклад=0.0439
chain1_p: λ=0.6816, ρ=0.1804, E[T]=0.3230, вклад=0.0835

Новое теоретическое среднее время E[T_sys] = 0.4554


## Новый YAML

In [16]:
def convert_to_python_types(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, dict):
        return {key: convert_to_python_types(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_python_types(item) for item in obj]
    else:
        return obj

new_config_clean = convert_to_python_types(new_config)
for t in new_config_clean['transitions']:
    t['probability'] = round(t['probability'], 8)

import yaml
new_yaml_str = yaml.dump(new_config_clean, sort_keys=False, allow_unicode=True, default_flow_style=False)
print("=== Новая конфигурация (YAML) ===\n")
print(new_yaml_str)

with open('optimized_config.yaml', 'w') as f:
    f.write(new_yaml_str)
print("Конфигурация сохранена в 'optimized_config.yaml'")

=== Новая конфигурация (YAML) ===

servers:
- id: chain1
  mu: 5.27253214380333
  queue_type: FIFO
  lambda: 2.0076164935244005
  Cost: 14899.8
  Jobs: 2818
  Avg_queue_time: 0.205557
  Avg_service_time: 0.187578
  Utilization: 0.528594
- id: chain2
  mu: 5.83754313849087
  queue_type: PS
  Cost: 18038.5
  Jobs: 613
  Avg_queue_time: 0.0109543
  Avg_service_time: 0.162924
  Utilization: 0.0998724
- id: chain3
  mu: 4.549136880106992
  queue_type: FIFO
  Cost: 11347.3
  Jobs: 186
  Avg_queue_time: 0.00526057
  Avg_service_time: 0.223484
  Utilization: 0.041568
- id: chain4
  mu: 8.123987347260954
  queue_type: PS
  Cost: 33999.6
  Jobs: 853
  Avg_queue_time: 0.0118444
  Avg_service_time: 0.118478
  Utilization: 0.101062
- id: chain1_p
  mu: 3.777430788663129
  queue_type: PS
  lambda: 0.6286495783772643
transitions:
- from: chain1
  to: exit
  probability: 0.51314919
- from: chain1
  to: chain2
  probability: 0.19733961
- from: chain1
  to: chain4
  probability: 0.28951121
- from: chain

## Проверка корректности переходов

In [17]:
for srv in new_config_clean['servers']:
    sid = srv['id']
    outgoing = [t['probability'] for t in new_config_clean['transitions'] if t['from'] == sid]
    total = sum(outgoing)
    print(f"{sid}: сумма вероятностей переходов = {total:.6f} (exit = {1-total:.6f})")

chain1: сумма вероятностей переходов = 1.000000 (exit = -0.000000)
chain2: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain3: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain4: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain1_p: сумма вероятностей переходов = 1.000000 (exit = -0.000000)


## Итоговый вывод

In [18]:
print("="*60)
print("ОПТИМИЗАЦИЯ ЗАВЕРШЕНА")
print("="*60)
print(f"Исходная стоимость системы: {total_cost:.2f}")
print(f"Бюджет на новые серверы: {budget:.2f} (10%)")
print(f"Выбранный вариант: {best_desc}")
# Оценим фактическую стоимость добавленных серверов
_, added_actual = evaluate_config(new_config, total_cost, c_fifo, alpha_fifo, c_ps, alpha_ps)
print(f"Фактическая дополнительная стоимость: {added_actual:.2f}")
print(f"Теоретическое улучшение E[T_sys]: {E_T_sys:.4f} → {E_T_sys_new:.4f} ({(E_T_sys - E_T_sys_new)/E_T_sys*100:.2f}%)")
print("Новая конфигурация сохранена в 'optimized_config.yaml'")

ОПТИМИЗАЦИЯ ЗАВЕРШЕНА
Исходная стоимость системы: 78285.20
Бюджет на новые серверы: 7828.52 (10%)
Выбранный вариант: 1 сервер к chain1 (μ=3.7774, α=0.2385)
Фактическая дополнительная стоимость: 7828.52
Теоретическое улучшение E[T_sys]: 0.5543 → 0.4554 (17.84%)
Новая конфигурация сохранена в 'optimized_config.yaml'


![img_1.png](img_1.png)

## Реальный результат: 
```
Simulation results (max time: 1000):
Server 'chain1'
  Type: FIFO
  Cost: 14899.8
  Jobs: 2239
  Avg queue time: 0.113014
  Avg service time: 0.180828
  Utilization: 0.404874
Server 'chain1_p'
  Type: PS
  Cost: 8134.49
  Jobs: 686
  Avg queue time: 0.0509475
  Avg service time: 0.261501
  Utilization: 0.17939
Server 'chain2'
  Type: PS
  Cost: 18038.5
  Jobs: 677
  Avg queue time: 0.0198502
  Avg service time: 0.171488
  Utilization: 0.11615
Server 'chain3'
  Type: FIFO
  Cost: 11347.3
  Jobs: 227
  Avg queue time: 0.00914131
  Avg service time: 0.20207
  Utilization: 0.0465772
Server 'chain4'
  Type: PS
  Cost: 33999.6
  Jobs: 873
  Avg queue time: 0.016838
  Avg service time: 0.129713
  Utilization: 0.113239

Total system cost: 86419.7
Total tasks: 2558
Mean time: 0.421695

Linear hist:
*                                       
*                                       
*                                       
*                                       
* *                                     
* *                                     
* *                                     
* * *                                   
* * *                                   
* * *                                   
* * *                                   
* * * *                                 
* * * *                                 
* * * *                                 
* * * * *                               
* * * * *                               
* * * * * *                             
* * * * * * *                           
* * * * * * *                           
* * * * * * * * * *                     
---------------------------------------
0.00                                3.05

Log hist:
  * *                                   
* * *                                   
* * *                                   
* * *   *                               
* * *   *                               
* * *   *                               
* * * * * *                             
* * * * * *                             
* * * * * *                             
* * * * * * *                           
* * * * * * *                           
* * * * * * * *                         
* * * * * * * *                         
* * * * * * * * *                       
* * * * * * * * *                       
* * * * * * * * * *                     
* * * * * * * * * * *                   
* * * * * * * * * * *                   
* * * * * * * * * * * * *               
* * * * * * * * * * * * * * * *         
---------------------------------------
0.00                                2.02
```

Есть небольшой вылет за бюджет, а значит модель расчёта стоимости неточная